# Preprocesamiento Textual

In [2]:
!pip install PyMuPDF pandas

   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
    --------------------------------------- 0.3/18.4 MB ? eta -:--:--
   ---- ----------------------------------- 2.1/18.4 MB 7.9 MB/s eta 0:00:03
   --------- ------------------------------ 4.5/18.4 MB 9.5 MB/s eta 0:00:02
   --------------- ------------------------ 7.3/18.4 MB 10.7 MB/s eta 0:00:02
   ---------------------- ----------------- 10.5/18.4 MB 11.5 MB/s eta 0:00:01
   ------------------------------ --------- 13.9/18.4 MB 12.3 MB/s eta 0:00:01
   ------------------------------------- -- 17.3/18.4 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------  18.4/18.4 MB 12.8 MB/s eta 0:00:01
   ---------------------------------------- 18.4/18.4 MB 10.1 MB/s  0:00:01



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
import fitz
import os
import pandas as pd
import re

### Toma de datos de los PDF

In [41]:
# Ruta raíz donde están los apuntes PDF
root = r"Documentos_de_Inteligencia_artificial_GR 2\Apuntadores"

# Lista vacía para guardar todos los resultados
documentos = []

# Recorremos todos los archivos de la carpeta
for archivo in os.listdir(root):
    # Nos aseguramos de trabajar solo con archivos que terminen en .pdf
    if archivo.lower().endswith(".pdf"):
        
        # Creamos la ruta completa del archivo
        ruta_pdf = os.path.join(root, archivo)
        
        # Abrimos el PDF con PyMuPDF
        with fitz.open(ruta_pdf) as pdf:
            texto_total = ""
            
            # Recorremos cada página y extraemos su texto
            # Usamos flags para mejor extracción de texto con Unicode
            for pagina in pdf:
                # flags=0 es el modo más simple y limpio
                # También podemos usar flags=fitz.TEXT_PRESERVE_WHITESPACE
                texto_total += pagina.get_text("text", flags=0) + "\n"
        
        # Guardamos los datos extraídos en una estructura temporal (diccionario)
        documentos.append({
            "nombre_archivo": archivo,
            "ruta": ruta_pdf,
            "texto": texto_total.strip()  # Quitamos espacios innecesarios al inicio y final
        })

# Convertimos la lista de documentos en una tabla (DataFrame) para trabajar más fácil
df_docs = pd.DataFrame(documentos)

# Mostramos las primeras filas para verificar que funcionó
df_docs.head()


,nombre_archivo,ruta,texto
0,10_SEMANA_AI_20251007_1-222887296.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes IA Clase 7/10\nGianmarco Oporta P´erez...
1,10_SEMANA_AI_20251007_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Redes Neuronales Convolucionales y\nBackpropag...
2,10_SEMANA_AI_20251009_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes de clase #2\nLuis Felipe Calderón Pére...
3,11_Semana_AI_20251014_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes IA Clase 14/10/2025\nJuan Jim´enez Val...
4,11_Semana_AI_20251014_2.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,"Inteligencia Artificial\nApuntes Semana 11, Cl..."


### Extracción de metadatos

In [44]:
import unicodedata

# Expresión regular para capturar datos del nombre del archivo
# Formato esperado: #semana_SEMANA_AI_yyyyddmm_#apunte.pdf
# Ejemplo: 10_SEMANA_AI_20251007_1.pdf
patron_nombre = r"(\d+)_semana_ai_(\d{4})(\d{2})(\d{2})_(\d+)\.pdf"

# Lista donde guardaremos los nuevos registros enriquecidos
datos_finales = []

# Definir lista de autores conocidos (estudiantes) - YA SIN TILDES
autores_conocidos = [
    "acuna lopez rodolfo david",
    "araya ortega fabian enrique",
    "benavides villegas luis fernando",
    "brenes martinez david",
    "brenes reyes fernando daniel",
    "brenes torres isaac david",
    "calderon perez luis felipe",
    "campos cerdas mauricio alonso",
    "carranza jimenez kevin josue",
    "diaz barboza fabian esteban",
    "espinoza aguilar dario",
    "gomez brenes gerardo alberto",
    "gonzalez sanchez luis alfredo",
    "jimenez salgado joselyn priscilla",
    "jimenez valverde juan diego",
    "murillo campos ian david",
    "naranjo masis alex steven",
    "oporta perez gianmarco",
    "quesada rodriguez jose pablo",
    "quesada sanchez mariana",
    "rodriguez camacho kendall andres",
    "rodriguez cano juan pablo",
    "rojas chacon sahid edgardo",
    "rojas obando nelson armando",
    "rojas rojas javier alonso",
    "sanchez araya brandon emmanuel",
    "sanchez rojas andres",
    "urena bermudez andrey",
    "varela venegas julio josue",
    "vargas solis rafael guillermo",
    "vasquez concepcion ashley lizeth",
    "vega suazo eder jose"
]

# Crear variaciones para mejor detección (apellidos completos) - SIN TILDES
apellidos_autores = [
    "acuna", "araya", "benavides", "brenes", "calderon", "campos",
    "carranza", "diaz", "espinoza", "gomez", "gonzalez", "jimenez",
    "murillo", "naranjo", "oporta", "quesada", "rodriguez", "rojas",
    "sanchez", "urena", "varela", "vargas", "vasquez", "vega"
]

for i, fila in df_docs.iterrows():
    nombre = fila["nombre_archivo"]
    texto = fila["texto"]

    # ------------------------------------------------------
    # 1. EXTRAER DATOS DEL NOMBRE DEL ARCHIVO
    # ------------------------------------------------------
    match = re.search(patron_nombre, nombre.lower())
    if match:
        semana = int(match.group(1))
        anio = int(match.group(2))
        dia = int(match.group(3))
        mes = int(match.group(4))
        apunte = int(match.group(5))
        fecha = f"{anio}-{mes:02d}-{dia:02d}"
    else:
        semana = None
        fecha = None
        apunte = None

    # ------------------------------------------------------
    # 2. EXTRAER TÍTULO Y AUTOR
    # ------------------------------------------------------
    lineas = texto.splitlines()
    titulo = lineas[0].strip() if lineas else None

    autor = None
    email = None

    # Recorremos línea por línea el texto extraído del PDF
    for linea in lineas[:15]:  # Solo las primeras 15 líneas (encabezado)
        linea_limpia = linea.strip().lower()
        
        # Buscar correo electrónico
        if "@estudiantec.cr" in linea_limpia:
            email = linea.strip()
            
        # Detectar autor: buscar coincidencias con apellidos conocidos
        # y que contenga al menos 2 palabras (nombre y apellido)
        if not autor and len(linea_limpia.split()) >= 2:
            # Verificar si contiene apellidos de estudiantes
            for apellido in apellidos_autores:
                if apellido in linea_limpia:
                    # Verificar que no contenga palabras del profesor
                    if "pacheco" not in linea_limpia and "portuguez" not in linea_limpia:
                        # Verificar que no sea el título ni otras líneas de metadata
                        if "abstract" not in linea_limpia and "index terms" not in linea_limpia:
                            autor = linea.strip()
                            break

    # ------------------------------------------------------
    # 3. EXTRAER ABSTRACT (RESUMEN)
    # ------------------------------------------------------
    abstract = None
    abstract_match = re.search(r"Abstract[—\-\s]+(.*?)(?:Index Terms|I\.|$)", texto, re.DOTALL | re.IGNORECASE)
    if abstract_match:
        abstract = abstract_match.group(1).strip()

    # ------------------------------------------------------
    # 4. GUARDAR DATOS ORGANIZADOS
    # ------------------------------------------------------
    datos_finales.append({
        "semana": semana,
        "fecha": fecha,
        "apunte": apunte,
        "titulo": titulo,
        "autor": autor,
        "email": email,
        "abstract": abstract,
        "texto": texto.strip(),
        "fuente": nombre
    })

# Crear un nuevo DataFrame con los metadatos enriquecidos
df_docs_full = pd.DataFrame(datos_finales)

# Mostrar una vista previa de los primeros registros
df_docs_full.head()


,semana,fecha,apunte,titulo,autor,email,abstract,texto,fuente
0,NaN,None,NaN,Apuntes IA Clase 7/10,Gianmarco Oporta P´erez,gooporta@estudiantec.cr,El presente documento recopila los apuntes de ...,Apuntes IA Clase 7/10\nGianmarco Oporta P´erez...,10_SEMANA_AI_20251007_1-222887296.pdf
1,10.0,2025-07-10,1.0,Redes Neuronales Convolucionales y,None,rodolfoide69@estudiantec.cr,En este documento podr´a encontrar informaci´o...,Redes Neuronales Convolucionales y\nBackpropag...,10_SEMANA_AI_20251007_1.pdf
2,10.0,2025-09-10,1.0,Apuntes de clase #2,None,None,None,Apuntes de clase #2\nLuis Felipe Calderón Pére...,10_SEMANA_AI_20251009_1.pdf
3,11.0,2025-14-10,1.0,Apuntes IA Clase 14/10/2025,"como los filtros, campos receptivos, stride, p...",juand0908@estudiantec.cr,Este documento resume los conceptos clave vist...,Apuntes IA Clase 14/10/2025\nJuan Jim´enez Val...,11_Semana_AI_20251014_1.pdf
4,11.0,2025-14-10,2.0,Inteligencia Artificial,Luis Fernando Benavides Villegas,lubenavides@estudiantec.cr,Este documento recopila los apuntes de la clas...,"Inteligencia Artificial\nApuntes Semana 11, Cl...",11_Semana_AI_20251014_2.pdf


### Limpieza de datos

In [50]:
import unicodedata

def eliminar_tildes(texto):
    """
    Elimina tildes y acentos manejando:
    1. Unicode compuesto (é)
    2. Unicode descompuesto (e + ´)
    3. Caracteres literales de acento (´, `, ¨, ^, ~)
    4. Caracteres especiales resultantes (ı, ȷ, etc.)
    """
    if not texto or not isinstance(texto, str):
        return texto
    
    # Paso 1: Eliminar caracteres literales de acento que aparecen solos
    acentos_literales = ['´', '`', '¨', '^', '~', '¯', '˜', '¸', 'ˆ', '˙', '˚', '˝']
    for acento in acentos_literales:
        texto = texto.replace(acento, '')
    
    # Paso 2: Normalizar a NFD (descomponer caracteres acentuados)
    texto_nfd = unicodedata.normalize('NFD', texto)
    
    # Paso 3: Filtrar marcas diacríticas (categoría Mn)
    texto_sin_tildes = ''.join(
        char for char in texto_nfd 
        if unicodedata.category(char) != 'Mn'
    )
    
    # Paso 4: Normalizar a NFC (recomponer)
    texto_final = unicodedata.normalize('NFC', texto_sin_tildes)
    
    # Paso 5: Reemplazar caracteres especiales que quedan
    # ı (dotless i) -> i
    # ȷ (dotless j) -> j
    # ø (o con barra) -> o
    # etc.
    reemplazos_especiales = {
        'ı': 'i',  # Latin small letter dotless i
        'ȷ': 'j',  # Latin small letter dotless j
        'ø': 'o',  # Latin small letter o with stroke
        'Ø': 'O',  # Latin capital letter O with stroke
        'ł': 'l',  # Latin small letter l with stroke
        'Ł': 'L',  # Latin capital letter L with stroke
        'ð': 'd',  # Latin small letter eth
        'Ð': 'D',  # Latin capital letter ETH
        'þ': 'th', # Latin small letter thorn
        'Þ': 'Th', # Latin capital letter THORN
        'ß': 'ss', # Latin small letter sharp s
    }
    
    for especial, normal in reemplazos_especiales.items():
        texto_final = texto_final.replace(especial, normal)
    
    return texto_final

def limpiar_campo_simple(texto):
    """Limpia campos individuales (autor, email, título)"""
    if not texto or not isinstance(texto, str):
        return texto
    
    # 1. Eliminar tildes
    texto = eliminar_tildes(texto)
    
    # 2. Convertir a minúsculas
    texto = texto.lower()
    
    # 3. Normalizar espacios
    texto = " ".join(texto.split())
    
    return texto

def limpiar_texto(texto):
    """Limpia campos de texto largo (abstract, texto completo)"""
    if not texto or not isinstance(texto, str):
        return texto
    
    # 1. Eliminar tildes
    texto = eliminar_tildes(texto)
    
    # 2. Convertir a minúsculas
    texto = texto.lower()
    
    # 3. Normalizar espacios y saltos de línea
    texto = " ".join(texto.split())
    
    return texto

def validar_autor(autor):
    """Valida si el autor detectado está en la lista de autores conocidos"""
    if not autor or not isinstance(autor, str):
        return autor
    
    autor_limpio = limpiar_campo_simple(autor)
    
    # Verificar si está en la lista de autores conocidos
    for nombre_conocido in autores_conocidos:
        if nombre_conocido in autor_limpio or autor_limpio in nombre_conocido:
            return autor
    
    return autor  # Si no se encuentra, mantener el original

# Aplicar limpieza a todos los campos del DataFrame
df_docs_full["titulo"] = df_docs_full["titulo"].apply(limpiar_campo_simple)
df_docs_full["autor"] = df_docs_full["autor"].apply(lambda x: validar_autor(x)).apply(limpiar_campo_simple)
df_docs_full["email"] = df_docs_full["email"].apply(limpiar_campo_simple)
df_docs_full["abstract"] = df_docs_full["abstract"].apply(limpiar_texto)
df_docs_full["texto"] = df_docs_full["texto"].apply(limpiar_texto)
df_docs_full["fuente"] = df_docs_full["fuente"].apply(limpiar_campo_simple)

# Mostrar todas las columnas (incluyendo texto limpio)
df_docs_full.head()


,semana,fecha,apunte,titulo,autor,email,abstract,texto,fuente
0,NaN,None,NaN,apuntes ia clase 7/10,gianmarco oporta perez,gooporta@estudiantec.cr,el presente documento recopila los apuntes de ...,apuntes ia clase 7/10 gianmarco oporta perez i...,10_semana_ai_20251007_1-222887296.pdf
1,10.0,2025-07-10,1.0,redes neuronales convolucionales y,None,rodolfoide69@estudiantec.cr,en este documento podra encontrar informacion ...,redes neuronales convolucionales y backpropaga...,10_semana_ai_20251007_1.pdf
2,10.0,2025-09-10,1.0,apuntes de clase #2,None,None,None,apuntes de clase #2 luis felipe calderon perez...,10_semana_ai_20251009_1.pdf
3,11.0,2025-14-10,1.0,apuntes ia clase 14/10/2025,"como los filtros, campos receptivos, stride, p...",juand0908@estudiantec.cr,este documento resume los conceptos clave vist...,apuntes ia clase 14/10/2025 juan jimenez valve...,11_semana_ai_20251014_1.pdf
4,11.0,2025-14-10,2.0,inteligencia artificial,luis fernando benavides villegas,lubenavides@estudiantec.cr,este documento recopila los apuntes de la clas...,"inteligencia artificial apuntes semana 11, cla...",11_semana_ai_20251014_2.pdf


### congelar dataframe en un .parquet

### Guardar DataFrame procesado

In [58]:
!pip install -U pandas pyarrow fastparquet

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
    --------------------------------------- 0.3/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 1.2 MB/s eta 0:00:09
   ----- ---------------------------------- 1.6/11.0 MB 3.0 MB/s eta 0:00:04
   ---------- ----------------------------- 2.9/11.0 MB 3.9 MB/s eta 0:00:03
   -------------- ------------------------- 3.9/11.0 MB 4.2 MB/s eta 0:00:02
   ------------------ --------------------- 5.0/11.0 MB 4.2 MB/s eta 0:00:02
   ---------------------- ----------------- 6.3/11.0 MB 4.5 MB/s eta 0:00:02
   --------------------------- ------------ 7.6/11.0 MB 4.7 MB/s eta 0:00:01
   ---------------------------------- ----- 9.4/11.0 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------  10.7/11.0 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------  10.7/11.0 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 4.7 MB/s  0:00:02
   ----------

  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
# Guardar el DataFrame limpio en formato Parquet
# Esto permite versionarlo y cargarlo rápidamente después
output_path = "df_docs_procesado.parquet"

try:
    # Intentar guardar con pyarrow
    df_docs_full.to_parquet(output_path, engine='pyarrow', compression='snappy')
    print(f"DataFrame guardado exitosamente en: {output_path}")
except Exception as e:
    # Si falla, usar pickle como alternativa
    print(f"Error con parquet: {e}")
    print("Guardando con pickle como alternativa...")
    output_path = "df_docs_procesado.pkl"
    df_docs_full.to_pickle(output_path)
    print(f"DataFrame guardado exitosamente en: {output_path}")

print(f"Total de documentos: {len(df_docs_full)}")
print(f"Columnas guardadas: {list(df_docs_full.columns)}")

Error con parquet: A type extension with name pandas.period already defined
Guardando con pickle como alternativa...
DataFrame guardado exitosamente en: df_docs_procesado.pkl
Total de documentos: 46
Columnas guardadas: ['semana', 'fecha', 'apunte', 'titulo', 'autor', 'email', 'abstract', 'texto', 'fuente']


In [ ]:
# Para cargar el DataFrame guardado en el futuro:
# df_docs_full = pd.read_pickle("df_docs_procesado.pkl")
# print(f"✅ DataFrame cargado: {len(df_docs_full)} documentos")

In [64]:
import os
import pandas as pd

def guardar_parquet_con_fallback(df, ruta):
    try:
        import pyarrow as pa  # asegúrate de importarlo aquí
        try:
            pa.unregister_extension_type("pandas.period")
        except Exception:
            pass
        df.to_parquet(ruta, index=False, engine="pyarrow", compression="snappy")
        print(f"Guardado con pyarrow en: {ruta}")
    except Exception as e:
        print("pyarrow falló:", e, "\n→ Intentando con fastparquet…")
        df.to_parquet(ruta, index=False, engine="fastparquet", compression="snappy")
        print(f"Guardado con fastparquet en: {ruta}")

carpeta_salida = "data"
os.makedirs(carpeta_salida, exist_ok=True)
ruta_parquet = os.path.join(carpeta_salida, "apuntes_clean_v1.parquet")

guardar_parquet_con_fallback(df_docs_full, ruta_parquet)

pyarrow falló: A type extension with name pandas.interval already defined 
→ Intentando con fastparquet…
Guardado con fastparquet en: data\apuntes_clean_v1.parquet


### Tecnicas de segmentación

#### Fixed-size Chunking with sliding window

In [66]:
# Fixed-size chunking con ventana deslizante (caracteres)
# Contrato:
# - Entrada: dataset limpio cargado desde data/*.parquet (sin modificar el DF original)
# - Salida: df_chunks_sliding con un chunk por fila, preservando metadatos útiles
# - Regla: stride = chunk_size - overlap

import os
import pandas as pd
from typing import List, Dict

# 1) Loader robusto: lee el dataset "congelado" desde /data
#    Intenta pyarrow -> fastparquet -> pickle de respaldo

def load_dataset():
    candidatos = [
        os.path.join("data", "apuntes_clean_v1.parquet"),
        os.path.join("data", "df_docs_procesado.parquet"),
    ]
    for ruta in candidatos:
        if os.path.exists(ruta):
            # intentar pyarrow, luego fastparquet
            try:
                return pd.read_parquet(ruta, engine="pyarrow")
            except Exception:
                try:
                    return pd.read_parquet(ruta, engine="fastparquet")
                except Exception:
                    pass
    # Fallback a pickle si existe
    pkl = os.path.join("data", "df_docs_procesado.pkl")
    if os.path.exists(pkl):
        return pd.read_pickle(pkl)
    # Último recurso: si existe en la raíz (caso previo)
    if os.path.exists("df_docs_procesado.pkl"):
        return pd.read_pickle("df_docs_procesado.pkl")
    raise FileNotFoundError(
        "No se encontró el dataset en data/*.parquet ni el pickle de respaldo."
    )


def chunk_text_sliding_window(texto: str, chunk_size: int = 1000, overlap: int = 200) -> List[Dict]:
    """
    Divide un texto en trozos (chunks) de tamaño fijo usando una ventana deslizante.
    - chunk_size: tamaño máximo del chunk en caracteres.
    - overlap: cantidad de caracteres que se solapan entre chunks consecutivos.
    Retorna una lista de dicts con: start, end, length, chunk, chunk_index.
    """
    if not isinstance(texto, str) or not texto:
        return []

    if chunk_size <= 0:
        raise ValueError("chunk_size debe ser > 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap debe estar en [0, chunk_size)")

    stride = chunk_size - overlap
    chunks: List[Dict] = []

    n = len(texto)
    idx = 0
    start = 0
    while start < n:
        end = min(start + chunk_size, n)
        fragmento = texto[start:end]
        if fragmento.strip():
            chunks.append({
                "chunk_index": idx,
                "start": start,
                "end": end,
                "length": end - start,
                "chunk": fragmento,
            })
            idx += 1
        if end == n:
            break
        start += stride

    return chunks

# 2) Cargar dataset desde /data sin tocar el DF original
pd.set_option('display.max_colwidth', None)
df_src = load_dataset()  # columnas esperadas: semana, fecha, apunte, titulo, autor, email, abstract, texto, fuente

# 3) Aplicar segmentación a todo el dataset y construir un DataFrame de chunks
registros = []
for row in df_src.itertuples(index=False):
    texto_doc = getattr(row, "texto")
    fuente = getattr(row, "fuente")
    for ch in chunk_text_sliding_window(texto_doc, chunk_size=1000, overlap=200):
        registros.append({
            "fuente": fuente,
            "semana": getattr(row, "semana", None),
            "fecha": getattr(row, "fecha", None),
            "apunte": getattr(row, "apunte", None),
            "titulo": getattr(row, "titulo", None),
            "autor": getattr(row, "autor", None),
            "chunk_index": ch["chunk_index"],
            "start": ch["start"],
            "end": ch["end"],
            "length": ch["length"],
            "chunk": ch["chunk"],
        })

df_chunks_sliding = pd.DataFrame(registros).sort_values(["fuente", "chunk_index"]).reset_index(drop=True)

# 4) Resumen y vista
num_docs = df_src.shape[0]
num_chunks = df_chunks_sliding.shape[0]
print(f" Documentos: {num_docs} | Chunks generados: {num_chunks} | Promedio por doc: {num_chunks / max(num_docs,1):.2f}")

df_chunks_sliding.head(10)

# 5) (Opcional) Guardar a disco para versionar resultados de segmentación
try:
    carpeta_salida = "data"
    os.makedirs(carpeta_salida, exist_ok=True)
    ruta_chunks = os.path.join(carpeta_salida, "chunks_sliding_v1.parquet")
    # Reusar el helper si existe en el notebook, sino guardar con parquet simple
    try:
        guardar_parquet_con_fallback(df_chunks_sliding, ruta_chunks)
    except NameError:
        # Si no está definida la función auxiliar, intentamos parquet directo con fallback simple
        try:
            df_chunks_sliding.to_parquet(ruta_chunks, engine="pyarrow", index=False, compression="snappy")
            print(f"Chunks guardados en: {ruta_chunks}")
        except Exception:
            alt = os.path.join(carpeta_salida, "chunks_sliding_v1.pkl")
            df_chunks_sliding.to_pickle(alt)
            print(f"Chunks guardados (pickle) en: {alt}")
except Exception as e:
    print("No se guardó parquet de chunks (opcional):", e)

 Documentos: 46 | Chunks generados: 620 | Promedio por doc: 13.48
pyarrow falló: A type extension with name pandas.interval already defined 
→ Intentando con fastparquet…
Guardado con fastparquet en: data\chunks_sliding_v1.parquet


#### Recursive chunking (Recursive Text Splitter)

Este método intenta mantener límites “naturales” del texto antes de forzarlo a tamaños fijos:
- Empieza dividiendo por separadores más fuertes a más débiles (párrafos → líneas → oraciones → signos → espacios).
- Si un fragmento sigue siendo largo, vuelve a dividirlo recursivamente con el siguiente separador.
- Al final, recombina piezas en chunks de tamaño objetivo con un solapamiento (overlap) definido para preservar contexto entre chunks.

Ventajas:
- Chunks más “semánticos” (menos cortes bruscos a mitad de ideas).
- Control fino del tamaño y del solapamiento.

Parámetros típicos:
- chunk_size: 1000 caracteres
- overlap: 200 caracteres
- separadores: ["\n\n", "\n", ". ", "; ", ", ", " "] (de fuerte → débil)

In [68]:
# Segmentación recursiva desde data/apuntes_clean_v1.parquet y guardado en data/chunks_recursive_v1.parquet
import os
import re
import pandas as pd
from typing import List

# Reutilizamos el loader robusto (pyarrow -> fastparquet -> pickle)
def load_dataset():
    rutas_intentos = [
        os.path.join('data', 'apuntes_clean_v1.parquet'),
        'apuntes_clean_v1.parquet',
    ]
    ultimo_error = None
    for ruta in rutas_intentos:
        if os.path.exists(ruta):
            try:
                try:
                    return pd.read_parquet(ruta, engine='pyarrow')
                except Exception as e1:
                    print(f"pyarrow falló: {e1}\n→ Intentando con fastparquet…")
                    try:
                        return pd.read_parquet(ruta, engine='fastparquet')
                    except Exception as e2:
                        ultimo_error = (e1, e2)
                        print(f"fastparquet también falló: {e2}")
            except Exception as e:
                ultimo_error = e
    # Fallback a pickle si existe
    for ruta in ['df_docs_procesado.pkl', os.path.join('data','df_docs_procesado.pkl')]:
        if os.path.exists(ruta):
            print("Cargando dataset desde pickle de respaldo…")
            return pd.read_pickle(ruta)
    raise RuntimeError(f"No se pudo cargar el dataset. Último error: {ultimo_error}")

# Utilidades: normalizar espacios y asegurar strings
_ws_re = re.compile(r"\s+")

def _s(text):
    if text is None:
        return ""
    return str(text)

# Split recursivo por una lista jerárquica de separadores
# separadores: de fuerte → débil (p. ej. párrafos → líneas → oraciones → espacios)

def recursive_split(text: str, separators: List[str], max_len: int) -> List[str]:
    text = _s(text).strip()
    if not text:
        return []
    # Caso base: si ya es corto, devolver tal cual
    if len(text) <= max_len:
        return [text]

    if not separators:
        # Sin separadores disponibles: forzar corte duro
        return [text[i:i+max_len] for i in range(0, len(text), max_len)]

    sep = separators[0]
    rest = separators[1:]

    # Dividir por el separador actual; si el separador es espacio simple, usar split(' ')
    if sep == ' ':
        parts = text.split(' ')
        glue = ' '
    else:
        parts = text.split(sep)
        glue = sep

    # Si el separador no generó cortes (texto sin ese separador), avanzar al siguiente
    if len(parts) == 1:
        return recursive_split(text, rest, max_len)

    # Repartir recursivamente cada parte si excede tamaño
    chunks = []
    current = []
    current_len = 0

    def flush_current():
        nonlocal current, current_len
        if current:
            combined = glue.join(current).strip()
            if combined:
                if len(combined) <= max_len:
                    chunks.append(combined)
                else:
                    # Aún grande: bajar a siguiente separador
                    chunks.extend(recursive_split(combined, rest, max_len))
        current = []
        current_len = 0

    for p in parts:
        piece = p.strip()
        if not piece:
            # Conserva separador donde sea útil; evitamos cadenas vacías consecutivas
            if current and glue:
                # Empuja un separador lógico en la recombinación
                current.append('')
            continue
        prospective_len = (current_len + (len(glue) if current else 0) + len(piece))
        if prospective_len <= max_len:
            # Aún cabe en el paquete actual
            if current:
                current.append(piece)
                current_len = prospective_len
            else:
                current = [piece]
                current_len = len(piece)
        else:
            # Cierra paquete actual y decide sobre piece
            flush_current()
            if len(piece) <= max_len:
                current = [piece]
                current_len = len(piece)
            else:
                # La pieza sola ya excede: desciende de nivel
                chunks.extend(recursive_split(piece, rest, max_len))
                current = []
                current_len = 0

    flush_current()
    return chunks

# Recombina listas de trozos cortos en ventanas con solapamiento, manteniendo tamaño objetivo

def pack_with_overlap(frags: List[str], chunk_size: int, overlap: int) -> List[str]:
    if not frags:
        return []
    # Normaliza y filtra vacíos
    norm = []
    for f in frags:
        s = _ws_re.sub(' ', _s(f)).strip()
        if s:
            norm.append(s)
    if not norm:
        return []

    chunks = []
    buf = []
    buf_len = 0

    def emit():
        nonlocal buf, buf_len
        if buf:
            out = ' '.join(buf).strip()
            if out:
                chunks.append(out)
        buf = []
        buf_len = 0

    for frag in norm:
        if buf_len == 0:
            buf.append(frag)
            buf_len = len(frag)
            continue
        prospective = buf_len + 1 + len(frag)  # +1 por espacio
        if prospective <= chunk_size:
            buf.append(frag)
            buf_len = prospective
        else:
            # Emite actual y aplica solapamiento aproximado por palabras
            emit()
            if overlap > 0 and chunks:
                tail = chunks[-1]
                # Toma ~overlap caracteres desde el final, cortado por palabras
                # Luego extrae últimas palabras para reconstruir contexto
                if len(tail) > overlap:
                    tail_ctx = tail[-overlap:]
                    # Evita iniciar en medio de palabra
                    tail_ctx = tail_ctx[tail_ctx.find(' ')+1:] if ' ' in tail_ctx else tail_ctx
                    if tail_ctx:
                        buf = [tail_ctx]
                        buf_len = len(tail_ctx)
                    else:
                        buf = []
                        buf_len = 0
                else:
                    buf = [tail]
                    buf_len = len(tail)
            else:
                buf = []
                buf_len = 0
            # Añade el fragmento actual (puede iniciar nuevo buffer)
            if buf_len == 0:
                buf = [frag]
                buf_len = len(frag)
            else:
                prospective = buf_len + 1 + len(frag)
                if prospective <= chunk_size:
                    buf.append(frag)
                    buf_len = prospective
                else:
                    emit()
                    buf = [frag]
                    buf_len = len(frag)

    emit()
    return chunks

# Pipeline: carga → split recursivo → empaquetado con solapamiento → DataFrame y guardado

df_docs_frozen = load_dataset()
assert 'texto' in df_docs_frozen.columns, "El dataset cargado debe contener la columna 'texto'"

separators = ["\n\n", "\n", ". ", "; ", ", ", " "]
chunk_size = 1000
overlap = 200

registros = []
for idx, row in df_docs_frozen.iterrows():
    texto = _s(row.get('texto', ''))
    if not texto.strip():
        continue
    # 1) Split recursivo por jerarquía de separadores
    frags = recursive_split(texto, separators, max_len=chunk_size)
    # 2) Empaquetar con solapamiento para uniformar tamaño objetivo
    chunks = pack_with_overlap(frags, chunk_size=chunk_size, overlap=overlap)

    for c_idx, c in enumerate(chunks):
        registros.append({
            'fuente': row.get('fuente'),
            'semana': row.get('semana'),
            'fecha': row.get('fecha'),
            'apunte': row.get('apunte'),
            'titulo': row.get('titulo'),
            'autor': row.get('autor'),
            'chunk_id': f"rec_{idx}_{c_idx}",
            'chunk_text': c,
            'chunk_len': len(c),
        })

# DataFrame de salida
chunks_recursive_df = pd.DataFrame(registros)
print(f" Documentos: {len(df_docs_frozen)} | Chunks generados (rec): {len(chunks_recursive_df)} | Promedio por doc: {len(chunks_recursive_df)/max(1,len(df_docs_frozen)):.2f}")
print(chunks_recursive_df['chunk_len'].describe().round(1))

# Guardado robusto en data/chunks_recursive_v1.parquet
os.makedirs('data', exist_ok=True)
salida = os.path.join('data', 'chunks_recursive_v1.parquet')
try:
    chunks_recursive_df.to_parquet(salida, engine='pyarrow', index=False)
    print(f"Guardado con pyarrow en: {salida}")
except Exception as e1:
    print(f"pyarrow falló: {e1}\n→ Intentando con fastparquet…")
    try:
        chunks_recursive_df.to_parquet(salida, engine='fastparquet', index=False)
        print(f"Guardado con fastparquet en: {salida}")
    except Exception as e2:
        print(f"fastparquet también falló: {e2}\n→ Guardando respaldo pickle…")
        salida_pkl = os.path.join('data', 'chunks_recursive_v1.pkl')
        chunks_recursive_df.to_pickle(salida_pkl)
        print(f"Respaldo guardado en: {salida_pkl}")

pyarrow falló: A type extension with name pandas.period already defined
→ Intentando con fastparquet…
 Documentos: 46 | Chunks generados (rec): 988 | Promedio por doc: 21.48
count     988.0
mean      595.1
std       361.9
min       180.0
25%       196.0
50%       836.5
75%       943.2
max      1000.0
Name: chunk_len, dtype: float64
pyarrow falló: A type extension with name pandas.period already defined
→ Intentando con fastparquet…
Guardado con fastparquet en: data\chunks_recursive_v1.parquet
